In [4]:
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q ollama

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 33 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (602 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 127332 files and directo

In [5]:
!nohup ollama serve > ollama.log 2>&1 &

In [6]:
import subprocess, time, ollama

subprocess.Popen(["ollama", "serve"])

for _ in range(30):          # chờ server sẵn sàng
    try:
        ollama.list()
        break
    except Exception:
        time.sleep(1)

ollama.pull("qwen2.5:7b")
print("Sẵn sàng")

Sẵn sàng


In [7]:
import ollama, json

MODEL = "qwen2.5:7b"

In [9]:
# ---------- 1. Các hàm thật ----------
PRODUCTS = {"iphone": 20000000, "laptop abc": 25000000, "keyboard": 500000, "penrcent": 10}
STOCK = {"iphone": 5, "laptop abc": 0, "keyboard": 12}


def search_product(product_name):
    price = PRODUCTS.get(product_name.lower())
    if price is None:
        return {"found": False, "message": "Không tìm thấy sản phẩm"}
    return {"found": True, "product": product_name, "price_vnd": price}

def check_stock(product_name):
    qty = STOCK.get(product_name.lower())
    if qty is None:
        return {"found": False, "message": "Không tìm thấy sản phẩm"}
    return {"found": True, "product": product_name, "quantity": qty}

def calculate_total(product_name, quantity):
    price = PRODUCTS.get(product_name.lower())
    if price is None:
        return {"found": False, "message": "Không tìm thấy sản phẩm"}
    return {"product": product_name, "quantity": quantity, "total_vnd": price * quantity}

def apply_discount(product_name, percent):
    price = PRODUCTS.get(product_name.lower())
    if price is None:
        return {"found": False, "message": "Không tìm thấy sản phẩm"}
    discounted_price = price * (1 - percent / 100)
    return {"product": product_name, "original_price": price, "discount_percent": percent, "discounted_price_vnd": int(discounted_price)}

available_functions = {
    "search_product": search_product,
    "check_stock": check_stock,
    "calculate_total": calculate_total,
    "apply_discount": apply_discount,
}


In [10]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_product",
            "description": "Tra giá (đồng) của một sản phẩm. Dùng khi người dùng hỏi giá.",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_name": {
                        "type": "string",
                        "description": "Tên sản phẩm"
                    }
                },
                "required": ["product_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_stock",
            "description": "Kiểm tra số lượng tồn kho của sản phẩm. Dùng khi hỏi còn hàng không, còn bao nhiêu.",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_name": {
                        "type": "string",
                        "description": "Tên sản phẩm"
                    }
                },
                "required": ["product_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_total",
            "d-": "Tính tổng tiền khi mua nhiều sản phẩm cùng loại. Dùng khi người dùng nói mua bao nhiêu cái.",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_name": {
                        "type": "string",
                        "description": "Tên sản phẩm"
                    },
                    "quantity": {
                        "type": "integer",
                        "description": "Số lượng mua"
                    }
                },
                "required": ["product_name", "quantity"]
            }
        }
    },
    {
    "type": "function",
    "function": {
        "name": "apply_discount",
        "description": "Tính giá tiền sau khi áp dụng phần trăm giảm giá cho sản phẩm.",
        "parameters": {
            "type": "object",
            "properties": {
                "product_name": {
                    "type": "string",
                    "description": "Tên sản phẩm "
                },
                "percent": {
                    "type": "integer",
                    "description": "Phần trăm giảm giá (ví dụ: 10 cho 10%)"
                }
            },
            "required": ["product_name", "percent"]
        }
    }
}
]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [11]:
# ---------- 3. Chạy tool an toàn ----------
def run_tool(call):
    name = call.function.name
    if name not in available_functions:
        return {"error": f"Tool '{name}' không tồn tại"}
    try:
        return available_functions[name](**call.function.arguments)
    except Exception as e:
        return {"error": f"Lỗi khi chạy tool: {e}"}

In [13]:

# ---------- 4. Hội thoại ----------
messages = [
    {"role": "system", "content": "Bạn là trợ lý bán hàng. Kết quả từ tool là dữ liệu chính xác. "
                                  "Chỉ dùng dữ liệu từ tool, không tự đoán."
                                  "Luôn Trả lời bằng tiếng Việt."},
    {"role": "user", "content": "Giá iphone là bao nhiêu và còn bao nhiêu keyboard, và cho mình hỏi thêm là sau đó giá iphone sau khi giảm 10% là bao nhiêu?"},
]

resp = ollama.chat(model=MODEL, messages=messages, tools=tools)

if not resp.message.tool_calls:
    print("Trả lời trực tiếp:", resp.message.content)
else:
    messages.append(resp.message)                         # lưu lượt model yêu cầu gọi tool

    for call in resp.message.tool_calls:                  # lặp qua TẤT CẢ lời gọi
        print("Function:", call.function.name, call.function.arguments)
        result = run_tool(call)
        print("Tool Result:", result)
        messages.append({"role": "tool", "name": call.function.name,
                         "content": json.dumps(result, ensure_ascii=False)})

    final = ollama.chat(model=MODEL, messages=messages, tools=tools)
    print("Trả lời:", final.message.content)

Function: search_product {'product_name': 'iphone'}
Tool Result: {'found': True, 'product': 'iphone', 'price_vnd': 20000000}
Function: check_stock {'product_name': 'keyboard'}
Tool Result: {'found': True, 'product': 'keyboard', 'quantity': 12}
Function: apply_discount {'product_name': 'iphone', 'percent': 10}
Tool Result: {'product': 'iphone', 'original_price': 20000000, 'discount_percent': 10, 'discounted_price_vnd': 18000000}
Trả lời: Giá của iPhone là 20,000,000 đồng. Hiện tại chúng tôi còn 12 chiếc keyboard trong kho. Sau khi áp dụng giảm giá 10%, giá của iPhone sẽ là 18,000,000 đồng.
